# Graph Analytics

Graph edge construction, centrality, communities, and Neo4j GDS playbook pointers.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.chdir(ROOT)
SAMPLE_ROWS = int(os.getenv("GTD_NOTEBOOK_SAMPLE_ROWS", "25000"))
print(f"Project root: {ROOT}")
print(f"Notebook sample rows: {SAMPLE_ROWS:,}")

Project root: C:\Users\kunmi\Personal Projects\Global-Terrorism-Database
Notebook sample rows: 25,000


In [2]:
import pandas as pd

from gtd_capstone.data.repository import DataRepository
from gtd_capstone.graph.analytics import (
    connected_components,
    degree_centrality,
    graph_edges_from_incidents,
    pagerank_baseline,
)
from gtd_capstone.graph.gds_playbook import gds_query_catalog

incidents = DataRepository().load_incidents(sample_rows=min(SAMPLE_ROWS, 3000))
edges = graph_edges_from_incidents(incidents)
{"incident_rows": len(incidents), "graph_edges": len(edges)}

{'incident_rows': 3000, 'graph_edges': 21000}

In [3]:
pd.DataFrame(degree_centrality(incidents, limit=10))

,node,degree,algorithm
0,Country:United States,1984,degree-centrality
1,Country:United Kingdom,1664,degree-centrality
2,WeaponType:Explosives,1458,degree-centrality
3,AttackType:Bombing/Explosion,1335,degree-centrality
4,Region:Western Europe,1298,degree-centrality
5,Region:North America,1026,degree-centrality
6,WeaponType:Firearms,897,degree-centrality
7,AttackType:Assassination,730,degree-centrality
8,TargetType:Business,691,degree-centrality
9,Year:1970,651,degree-centrality


In [4]:
pd.DataFrame(pagerank_baseline(incidents, iterations=5)).head(10)

,node,score,algorithm
0,WeaponType:Explosives,0.049606,pagerank-baseline
1,AttackType:Bombing/Explosion,0.045424,pagerank-baseline
2,Country:United States,0.033705,pagerank-baseline
3,WeaponType:Firearms,0.030302,pagerank-baseline
4,Country:United Kingdom,0.027857,pagerank-baseline
5,AttackType:Assassination,0.024600,pagerank-baseline
6,TargetType:Business,0.023517,pagerank-baseline
7,Year:1970,0.022172,pagerank-baseline
8,Year:1974,0.019871,pagerank-baseline
9,Year:1972,0.019169,pagerank-baseline


In [5]:
pd.DataFrame(connected_components(incidents, limit=5))

,community_id,size,sample_nodes,algorithm
0,0,3349,"[Incident:197001300003, WeaponType:Fake Weapon...",union-find-connected-components-baseline


In [6]:
gds = pd.DataFrame(gds_query_catalog())
gds["cypher_excerpt"] = gds["cypher"].str.slice(0, 140)
gds[["name", "cypher_excerpt"]].head(8)

,name,cypher_excerpt
0,projection,CALL gds.graph.project(\n 'gtd-knowledge-grap...
1,pagerank,"CALL gds.pageRank.write('gtd-knowledge-graph',..."
2,louvain,"CALL gds.louvain.write('gtd-knowledge-graph', ..."
3,node_similarity,CALL gds.nodeSimilarity.write('gtd-knowledge-g...
4,fastrp,"CALL gds.fastRP.write('gtd-knowledge-graph', {..."
5,country_profile,"MATCH (c:Country)\nRETURN c.name AS country, c..."
